In [ ]:
from typing import Optional

from openai import OpenAI
from openai.types.responses import Response
from rich.console import Console
from rich.panel import Panel

from settings import settings

In [ ]:
class APIUsageTracker:
    def __init__(self) -> None:
        self.prompt_tokens: int = 0
        self.output_tokens: int = 0
        self.total_tokens: int = 0
        self.total_cost: float = 0.0
        self.last_response: Optional[Response] = None
        self._console = Console()

    def add_response(self, response: Response) -> None:
        self.last_response = response
        usage = response.usage
        self.prompt_tokens += usage.input_tokens
        self.output_tokens += usage.output_tokens
        self.total_tokens += usage.total_tokens
        self.total_cost += usage.cost

    def get_summary(self) -> dict[str, int | float]:
        return {
            "input_tokens": self.prompt_tokens,
            "output_tokens": self.output_tokens,
            "total_tokens": self.total_tokens,
            "total_cost": self.total_cost,
        }

    def show(self) -> None:
        if self.last_response is None:
            self._console.print("No response available", style="red")
            return

        response_text = self.last_response.output_text or str(self.last_response)
        self._console.print(f"(Tokens: {self.total_tokens})", justify="left", style="cyan")
    def __str__(self) -> str:
        return (
            f"Tokens tracker - Entrada: {self.prompt_tokens}, Salida: {self.output_tokens}, "
            f"Total: {self.total_tokens}, Costo: {self.total_cost}"
        )


# Inicializamos el tracker en memoria
tracker = APIUsageTracker()

## Basic API call

In [25]:
client = OpenAI(
    base_url='https://openrouter.ai/api/v1',
    api_key=settings.open_router_key,
)

In [26]:
settings.open_router_model

'openai/gpt-oss-120b:free'

In [27]:
response = client.responses.create(
    model=settings.open_router_model,
    reasoning={'effort': 'low'},
    input='Hi, how are you? which model are you?',
)

tracker.add_response(response)

In [28]:
tracker.show()

                                                                                                      (Tokens: 130)

╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ Hello! I'm doing great, thanks for asking. I'm ChatGPT, the large language model powered by OpenAI's GPT‑4      │
│ architecture. How can I help you today?                                                                         │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

In [30]:
## Structured output
from pydantic import BaseModel

class CalendarEvent(BaseModel):
    title: str
    date: str
    time: str


In [35]:
response = client.responses.parse(
    model=settings.open_router_model,
    input=[
        {
            'role': 'system',
            'content': 'Extract the event information'
        },
        {
            'role': 'user',
            'content': 'september 11th 2001, was an incredible day'
        }
    ],
    text_format=CalendarEvent
)

tracker.add_response(response)

In [36]:
tracker.show()

                                                                                                      (Tokens: 460)

╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ {                                                                                                               │
│   "date": "2001-09-11",                                                                                         │
│   "time": "unspecified",                                                                                        │
│   "title": "September 11 attacks (commonly referred to as 9/11) – terrorist attacks on the United States,       │
│ including the destruction of the World Trade Center towers, the crash of Flight 93, and the attack on the       │
│ Pentagon."                                                                                                      │
│ }                                                                                                               │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

In [37]:
print(response.output_text)

{
  "date": "2001-09-11",
  "time": "unspecified",
  "title": "September 11 attacks (commonly referred to as 9/11) – terrorist attacks on the United States, including the destruction of the World Trade Center towers, the crash of Flight 93, and the attack on the Pentagon."
}
